# Kairos-CIDA · Notebook 4: Offline Latent Caching

Run the frozen 7B/8B LLM **once**, cache every hidden state to disk, then unload it.
Training CIDA in Notebook 5 never loads the LLM again — it reads cached vectors directly.

```
LLM (loaded once) -> pooled_hidden + prompt_embeds -> .arrow cache -> unload LLM
                                                             |
                                             Notebook 5 reads only these tensors
```

Datasets: **NuminaMath-CoT** (math chains) + **OpenOrca** (reasoning) + **PRM800K** (step labels)


## 0 · Setup


In [ ]:
import subprocess, sys, os
from getpass import getpass

# 1. Install dependencies
!pip install -q torch transformers datasets accelerate safetensors torchdiffeq

# 2. Securely clone the private repository using a Personal Access Token (PAT)
if not os.path.exists('CIDA-plugin'):
    print('This repository is private. Please enter your GitHub Personal Access Token (PAT):')
    pat = getpass('GitHub PAT: ').strip()
    clone_url = f'https://{pat}@github.com/Kairatzh/CIDA-plugin.git'
    try:
        subprocess.run(['git', 'clone', '--depth', '1', clone_url], check=True)
        print('CIDA-plugin cloned successfully.')
    except Exception as e:
        print('Error cloning repository. Please check your token and try again.', e)
else:
    print('CIDA-plugin already exists.')

sys.path.insert(0, os.path.abspath('CIDA-plugin'))

# 3. Mount Google Drive for persistent caching
from google.colab import drive
drive.mount('/content/drive')
print('Setup complete.')


## 1 · Configuration


In [ ]:
import gc, re, json, time, warnings
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Optional

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset, Dataset as HFDataset, concatenate_datasets
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

@dataclass
class CacheConfig:
    # Note: Qwen/Qwen2.5-7B-Instruct is a public, non-gated model that fits easily on a free T4 GPU
    model_name:        str   = 'Qwen/Qwen2.5-7B-Instruct'
    cache_dir:         str   = '/content/drive/MyDrive/Kairos_Cache'
    max_prompt_tokens: int   = 384
    max_answer_tokens: int   = 256
    numina_cap:        int   = 15000
    orca_cap:          int   = 10000
    prm_cap:           int   = 5000
    batch_size:        int   = 8
    save_every:        int   = 500

cfg = CacheConfig()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
Path(cfg.cache_dir).mkdir(parents=True, exist_ok=True)
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


## 2 · Load Frozen LLM + Save LM Head


In [ ]:
def flush():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f'Loading {cfg.model_name} ...')
tokenizer = AutoTokenizer.from_pretrained(
    cfg.model_name, trust_remote_code=True, padding_side='left')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

llm = AutoModelForCausalLM.from_pretrained(
    cfg.model_name, trust_remote_code=True,
    torch_dtype=torch.float16, device_map='auto',
    output_hidden_states=True,
    # Note: If running on a free Google Colab T4 GPU, you can uncomment load_in_8bit=True below to avoid VRAM OOM
    # load_in_8bit=True
)
for p in llm.parameters(): p.requires_grad = False
llm.eval()

LLM_HIDDEN  = llm.config.hidden_size
VOCAB_SIZE  = llm.config.vocab_size
embed_tok   = llm.get_input_embeddings()

# Save LM head separately so Notebook 5 can compute language loss
# without loading the full 7B/8B model
lm_head = llm.lm_head if hasattr(llm, 'lm_head') else llm.embed_out
torch.save(lm_head.state_dict(),
           str(Path(cfg.cache_dir) / 'lm_head.pt'))
# Also save the weight shape for reconstruction
lm_head_cfg = {'in_features': LLM_HIDDEN, 'out_features': VOCAB_SIZE}
with open(str(Path(cfg.cache_dir) / 'lm_head_cfg.json'), 'w') as f:
    json.dump(lm_head_cfg, f)

print(f'LLM loaded. hidden_size={LLM_HIDDEN}, vocab_size={VOCAB_SIZE}')
print(f'LM head saved to {cfg.cache_dir}/lm_head.pt')


## 3 · Dataset Loaders


In [ ]:
SYSTEMS = {
    'math':  'You are a precise mathematical reasoner. Solve step by step, end with #### <answer>.',
    'logic': 'You are a logical reasoning expert. Think step by step.',
    'mixed': 'You are a careful reasoner. Think step by step before answering.',
}

def make_prompt(q, task='math'):
    s = SYSTEMS.get(task, SYSTEMS['mixed'])
    return (f'<|im_start|>system\n{s}\n<|im_end|>\n'
            f'<|im_start|>user\n{q.strip()}\n<|im_end|>\n'
            f'<|im_start|>assistant\n')

def is_clean(t):
    return isinstance(t, str) and bool(t.strip()) and '\ufffd' not in t

MAX_STEPS = 8

def extract_steps_and_labels(solution_text):
    steps = [l.strip() for l in re.split(r'\n\n|Step \d+:', solution_text)
             if l.strip()]
    steps = steps[:MAX_STEPS]
    labels = [1.0 if re.search(r'[=+\-*/^]|\d+', st) else 0.5 for st in steps]
    has_ans = bool(re.search(r'####|\\boxed|therefore', solution_text[-200:]))
    if labels: labels[-1] = 1.0 if has_ans else 0.0
    # Pad to MAX_STEPS
    labels = (labels + [0.0] * MAX_STEPS)[:MAX_STEPS]
    steps  = (steps  + ['']  * MAX_STEPS)[:MAX_STEPS]
    return steps, labels


def load_numina(cap):
    try:
        # Load only a slice to avoid CPU RAM overflow in Colab
        raw = load_dataset('AI-MO/NuminaMath-CoT', split=f'train[:{cap * 2}]', trust_remote_code=True)
    except Exception:
        try:
            raw = load_dataset('lighteval/MATH', 'all', split=f'train[:{cap * 2}]', trust_remote_code=True)
        except Exception:
            print('  NuminaMath unavailable; skipping.'); return []
    out, skip = [], 0
    for ex in tqdm(raw, desc='NuminaMath', total=min(len(raw), cap)):
        if len(out) >= cap: break
        q = (ex.get('problem') or ex.get('problem') or '').strip()
        s = (ex.get('solution') or ex.get('solution') or '').strip()
        if not (is_clean(q) and is_clean(s)): skip += 1; continue
        steps, labels = extract_steps_and_labels(s)
        out.append({'prompt': make_prompt(q, 'math'), 'answer': s,
                    'steps': steps, 'step_labels': labels,
                    'task': 'math', 'source': 'numina'})
    print(f'  NuminaMath -> {len(out):>6} | {skip} skipped'); return out


def load_openorca(cap):
    try:
        # Load only a slice to avoid CPU RAM overflow in Colab
        raw = load_dataset('Open-Orca/OpenOrca', split=f'train[:{cap * 4}]', trust_remote_code=True)
    except Exception:
        print('  OpenOrca unavailable; skipping.'); return []
    out, skip = [], 0
    for ex in tqdm(raw, desc='OpenOrca', total=min(len(raw), cap * 3)):
        if len(out) >= cap: break
        q = (ex.get('question') or ex.get('question') or '').strip()
        a = (ex.get('response') or ex.get('response') or '').strip()
        if not (is_clean(q) and is_clean(a)): skip += 1; continue
        has_reasoning = any(kw in a.lower() for kw in
                            ['step', 'first', 'then', 'because', 'therefore', 'thus'])
        if not has_reasoning: skip += 1; continue
        steps, labels = extract_steps_and_labels(a)
        out.append({'prompt': make_prompt(q, 'mixed'), 'answer': a,
                    'steps': steps, 'step_labels': labels,
                    'task': 'mixed', 'source': 'openorca'})
    print(f'  OpenOrca   -> {len(out):>6} | {skip} skipped'); return out


def load_prm800k(cap):
    for ds_id in ['openai/prm800k',
                  'Birchlabs/openai-prm800k-stepwise-critic']:
        try:
            # Load only a slice to avoid CPU RAM overflow in Colab
            raw = load_dataset(ds_id, split=f'train[:{cap * 2}]', trust_remote_code=True); break
        except Exception: pass
    else:
        print('  PRM800K unavailable; skipping.'); return []
    out, skip = [], 0
    for ex in tqdm(raw, desc='PRM800K', total=min(len(raw), cap * 2)):
        if len(out) >= cap: break
        q = (ex.get('question') or ex.get('problem') or '').strip()
        steps_raw = ex.get('steps') or ex.get('completions') or []
        if not (is_clean(q) and steps_raw): skip += 1; continue
        steps, labels = [], []
        for step in (steps_raw if isinstance(steps_raw, list) else []):
            if isinstance(step, dict):
                text  = (step.get('text') or step.get('completion') or '').strip()
                label = float(step.get('label') or step.get('rating') or 0)
            else:
                text, label = str(step).strip(), 0.0
            if text:
                steps.append(text)
                labels.append(max(0.0, (label + 1) / 2))  # {-1,0,1} -> {0,0.5,1}
        if not steps: skip += 1; continue
        # Pad to MAX_STEPS
        labels = (labels + [0.0] * MAX_STEPS)[:MAX_STEPS]
        steps  = (steps  + ['']  * MAX_STEPS)[:MAX_STEPS]
        answer = ' '.join(s for s in steps if s)
        out.append({'prompt': make_prompt(q, 'math'), 'answer': answer,
                    'steps': steps, 'step_labels': labels,
                    'task': 'math', 'source': 'prm800k'})
    print(f'  PRM800K    -> {len(out):>6} | {skip} skipped'); return out

print('Dataset loaders defined.')


## 4 · Run Loaders


In [ ]:
import random
random.seed(42)

numina_recs = load_numina(cfg.numina_cap)
orca_recs   = load_openorca(cfg.orca_cap)
prm_recs    = load_prm800k(cfg.prm_cap)

all_records = numina_recs + orca_recs + prm_recs
random.shuffle(all_records)

src_counts = {}
for r in all_records: src_counts[r['source']] = src_counts.get(r['source'],0)+1
print(f'Total: {len(all_records):,}')
for s, c in sorted(src_counts.items()): print(f'  {s:<12} {c:>6,}')


## 5 · Offline Latent Caching

Each prompt is passed through the LLM **once**. We extract:

- `pooled_hidden` (LLM_HIDDEN,) — contextualised last-token state — this is what CIDA receives
- `prompt_embeds` (Lq, LLM_HIDDEN) — raw token embeddings — used for plan injection

After this cell the LLM is deleted. Notebook 5 never loads it.


In [ ]:
@torch.no_grad()
def extract_states(prompt_ids, prompt_mask):
    out    = llm(input_ids=prompt_ids, attention_mask=prompt_mask,
                 output_hidden_states=True, use_cache=False)
    last_h = out.hidden_states[-1]                       # (B, Lq, H)
    sidx   = prompt_mask.sum(1) - 1
    bidx   = torch.arange(prompt_ids.size(0), device=prompt_ids.device)
    pooled = last_h[bidx, sidx].cpu().float()            # (B, H)
    p_emb  = embed_tok(prompt_ids).cpu().float()         # (B, Lq, H)
    assert pooled.shape[-1] == LLM_HIDDEN
    assert p_emb.shape[1]   == prompt_ids.size(1)
    return pooled, p_emb

def tokenize_rec(rec):
    tokenizer.padding_side = 'left'
    p = tokenizer(rec['prompt'], max_length=cfg.max_prompt_tokens,
                  truncation=True, padding='max_length', return_tensors='pt')
    tokenizer.padding_side = 'right'
    a = tokenizer(rec['answer'], max_length=cfg.max_answer_tokens,
                  truncation=True, padding='max_length', return_tensors='pt')
    tokenizer.padding_side = 'left'
    sl = (rec.get('step_labels', []) + [0.0]*MAX_STEPS)[:MAX_STEPS]
    return {'pid': p['input_ids'].squeeze(0), 'pmsk': p['attention_mask'].squeeze(0),
            'aid': a['input_ids'].squeeze(0), 'amsk': a['attention_mask'].squeeze(0),
            'sl': torch.tensor(sl, dtype=torch.float32),
            'task': rec['task'], 'source': rec['source']}

# ── Main caching loop ────────────────────────────────────────────────────
cache_buffer, failed, shard_n = [], 0, 0
t0 = time.time()

for start in tqdm(range(0, len(all_records), cfg.batch_size), desc='Caching'):
    batch = all_records[start : start + cfg.batch_size]
    toks  = [tokenize_rec(r) for r in batch]

    pid  = torch.stack([t['pid']  for t in toks]).to(DEVICE)
    pmsk = torch.stack([t['pmsk'] for t in toks]).to(DEVICE)

    try:
        pooled_h, prompt_e = extract_states(pid, pmsk)
    except RuntimeError as e:
        failed += len(toks); flush(); continue

    for i, tok in enumerate(toks):
        cache_buffer.append({
            'pooled_hidden': pooled_h[i].numpy().tolist(),
            'prompt_embeds': prompt_e[i].numpy().tolist(),
            'answer_ids':    tok['aid'].numpy().tolist(),
            'answer_mask':   tok['amsk'].numpy().tolist(),
            'step_labels':   tok['sl'].numpy().tolist(),
            'task':          tok['task'],
            'source':        tok['source'],
            'hidden_dim':    LLM_HIDDEN,
            'prompt_len':    cfg.max_prompt_tokens,
            'answer_len':    cfg.max_answer_tokens,
        })

    if len(cache_buffer) >= cfg.save_every:
        sp = str(Path(cfg.cache_dir) / f'shard_{shard_n:05d}')
        HFDataset.from_list(cache_buffer).save_to_disk(sp)
        cache_buffer = []; shard_n += 1; flush()

if cache_buffer:
    HFDataset.from_list(cache_buffer).save_to_disk(
        str(Path(cfg.cache_dir) / f'shard_{shard_n:05d}'))

print(f'Done in {time.time()-t0:.0f}s  |  failed: {failed}')


## 6 · Merge Shards + Unload LLM


In [ ]:
from datasets import load_from_disk

shards = [load_from_disk(str(p))
          for p in sorted(Path(cfg.cache_dir).glob('shard_*'))]
full   = concatenate_datasets(shards)
split  = full.train_test_split(test_size=0.05, seed=42)
split['train'].save_to_disk(str(Path(cfg.cache_dir)/'train'))
split['test'].save_to_disk(str(Path(cfg.cache_dir)/'val'))

with open(str(Path(cfg.cache_dir)/'cache_meta.json'), 'w') as f:
    json.dump({'model_name': cfg.model_name, 'llm_hidden': LLM_HIDDEN,
               'vocab_size': VOCAB_SIZE, 'n_train': len(split['train']),
               'n_val': len(split['test']), 'sources': src_counts}, f, indent=2)

print(f'Train: {len(split["train"]):,}  |  Val: {len(split["test"]):,}')

# ── Unload LLM permanently ───────────────────────────────────────────────
del llm, embed_tok
flush()
if torch.cuda.is_available():
    print(f'VRAM after unload: {torch.cuda.memory_allocated()/1e9:.3f} GB')
print('LLM unloaded. Notebook 5 does not need it.')


## 7 · Cache Verification


In [ ]:
from datasets import load_from_disk

train_cache = load_from_disk(str(Path(cfg.cache_dir)/'train'))
s = train_cache[0]

ph = torch.tensor(s['pooled_hidden'])
pe = torch.tensor(s['prompt_embeds'])
ai = torch.tensor(s['answer_ids'])
sl = torch.tensor(s['step_labels'])

assert ph.shape == (LLM_HIDDEN,),                       f'pooled_hidden: {ph.shape}'
assert pe.shape == (cfg.max_prompt_tokens, LLM_HIDDEN), f'prompt_embeds: {pe.shape}'
assert ai.shape == (cfg.max_answer_tokens,),            f'answer_ids   : {ai.shape}'
assert sl.shape == (MAX_STEPS,),                        f'step_labels  : {sl.shape}'

print('All shape assertions passed.')
print(f'  pooled_hidden : {tuple(ph.shape)}  norm={ph.norm():.3f}')
print(f'  prompt_embeds : {tuple(pe.shape)}')
print(f'  answer_ids    : {tuple(ai.shape)}')
print(f'  step_labels   : {[round(x,2) for x in sl.tolist()]}')
print(f'  task / source : {s["task"]} / {s["source"]}')
print(f'\nCache ready. Open Notebook 5.')
